# Treccani validation of Cultura

## Settings

In [1]:
from pathlib import Path
import re

import duckdb
import pandas as pd

ROOT = Path.cwd().parent
AI_ANSWERS = ROOT / "annotations" / "treccani_validation" / "treccani_ai_annotations.parquet"
DATABASE = ROOT / "data" / "cultura" / "humans_clean_v2_sample_treccani.duckdb"
MIN_OVERLAP_SHARE = 0.5

## Treccani answers found by the AI

In [2]:
def window_years(answer):
    years = [int(year) for year in re.findall(r"\b\d{3,4}\b", answer or "")]
    if years:
        return min(years), max(years)
    centuries = [int(c) for c in re.findall(r"(\d{1,2})(?:st|nd|rd|th) century", answer or "")]
    if centuries:
        return (min(centuries) - 1) * 100 + 1, max(centuries) * 100
    return None, None


raw = pd.read_parquet(AI_ANSWERS)
treccani = pd.DataFrame({
    "qid": raw["entity"].map(lambda entity: entity["qid"]),
    "name": raw["entity"].map(lambda entity: entity["label_en"]),
    "treccani_polity": raw["polity_ai_matched"].map(lambda answer: answer["answer"]),
    "treccani_polity_id": raw["polity_ai_matched"].map(lambda answer: int(answer["source_verbatim"][0].split("|")[0]) if len(answer["source_verbatim"]) else None),
    "treccani_window": raw["productivity_window_ai_extracted"].map(lambda answer: answer["answer"]),
})
treccani[["treccani_start", "treccani_end"]] = treccani["treccani_window"].map(window_years).tolist()
assert treccani["qid"].is_unique
print(len(treccani), "individuals")
treccani

5 individuals


,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end
0,Q16268416,Domenico Giunti,Duchy of Milan,923,1523-1560,1523,1560
1,Q109861078,Olimpio Cozzolino,Kingdom of Italy,350,1894-1937,1894,1937
2,Q3427899,Restaino Cantelmo Stuart,Kingdom of Spain,1000,1675-1721,1675,1721
3,Q299458,Sibylla of Acerra,Kingdom of Sicily,680,1190-1204 or 1205,1190,1205
4,Q26246988,Nicola Paglia,Papal States,419,1220-1255,1220,1255


## Cultura window and polities

In [3]:
connection = duckdb.connect(str(DATABASE), read_only=True)
qids = treccani["qid"].tolist()

windows = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           individual_enriched.peak_productivity.start_year AS cultura_start,
           individual_enriched.peak_productivity.end_year AS cultura_end
    FROM individual_enriched
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
""", [qids]).df()
windows["cultura_window"] = windows.apply(lambda row: None if pd.isna(row["cultura_start"]) else f"{int(row['cultura_start'])}-{int(row['cultura_end'])}", axis=1)

sub_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(polity_match.polity.cliopatria_id) AS cultura_sub_polity_ids,
           string_agg(polity_match.polity.name, ' / ') AS cultura_sub_polities
    FROM individual_enriched, unnest(individual_enriched.polity) AS unnested(polity_match)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
    GROUP BY qid
""", [qids]).df()

meta_polities = connection.execute("""
    SELECT individual_enriched.entity.qid AS qid,
           list(DISTINCT meta.cliopatria_id) AS cultura_meta_polity_ids,
           string_agg(DISTINCT meta.name, ' / ') AS cultura_meta_polities
    FROM individual_enriched,
         unnest(individual_enriched.polity) AS unnested(polity_match),
         unnest(polity_match.polity.meta_polities) AS unnested_meta(meta)
    WHERE individual_enriched.entity.qid IN (SELECT unnest(?))
      AND meta.start_year <= individual_enriched.peak_productivity.end_year
      AND meta.end_year >= individual_enriched.peak_productivity.start_year
    GROUP BY qid
""", [qids]).df()

cultura = windows.merge(sub_polities, on="qid", how="left").merge(meta_polities, on="qid", how="left")
print(len(cultura), "individuals,", cultura["cultura_sub_polities"].notna().sum(), "with a polity")
cultura

5 individuals, 4 with a polity


,qid,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q109861078,1898,1922,1898-1922,[350],Kingdom of Italy,NaN,NaN
1,Q26246988,1238,1246,1238-1246,[523],Holy Roman Empire,[1612],Holy Roman Empire
2,Q3427899,1690,1707,1690-1707,[1000],Kingdom of Spain,[1622],Spanish Empire
3,Q16268416,1531,1560,1531-1560,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire
4,Q299458,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN


## Comparison table

In [4]:
comparison = treccani.merge(cultura, on="qid", how="left")
assert len(comparison) == len(treccani)
comparison

,qid,name,treccani_polity,treccani_polity_id,treccani_window,treccani_start,treccani_end,cultura_start,cultura_end,cultura_window,cultura_sub_polity_ids,cultura_sub_polities,cultura_meta_polity_ids,cultura_meta_polities
0,Q16268416,Domenico Giunti,Duchy of Milan,923,1523-1560,1523,1560,1531,1560,1531-1560,[783],Holy Roman Empire Minor States,[1612],Holy Roman Empire
1,Q109861078,Olimpio Cozzolino,Kingdom of Italy,350,1894-1937,1894,1937,1898,1922,1898-1922,[350],Kingdom of Italy,NaN,NaN
2,Q3427899,Restaino Cantelmo Stuart,Kingdom of Spain,1000,1675-1721,1675,1721,1690,1707,1690-1707,[1000],Kingdom of Spain,[1622],Spanish Empire
3,Q299458,Sibylla of Acerra,Kingdom of Sicily,680,1190-1204 or 1205,1190,1205,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
4,Q26246988,Nicola Paglia,Papal States,419,1220-1255,1220,1255,1238,1246,1238-1246,[523],Holy Roman Empire,[1612],Holy Roman Empire


## Polity accuracy

In [5]:
as_list = lambda ids: [] if ids is None or isinstance(ids, float) else list(ids)
sub_ids = comparison["cultura_sub_polity_ids"].map(as_list)
meta_ids = comparison["cultura_meta_polity_ids"].map(as_list)
comparison["polity_match"] = [
    "sub polity" if treccani_id in subs else "meta polity" if treccani_id in metas else "none"
    for treccani_id, subs, metas in zip(comparison["treccani_polity_id"], sub_ids, meta_ids)
]
comparison["polity_ok"] = comparison["polity_match"] != "none"
comparison[["name", "treccani_polity", "cultura_sub_polities", "cultura_meta_polities", "polity_match", "polity_ok"]]

,name,treccani_polity,cultura_sub_polities,cultura_meta_polities,polity_match,polity_ok
0,Domenico Giunti,Duchy of Milan,Holy Roman Empire Minor States,Holy Roman Empire,none,False
1,Olimpio Cozzolino,Kingdom of Italy,Kingdom of Italy,NaN,sub polity,True
2,Restaino Cantelmo Stuart,Kingdom of Spain,Kingdom of Spain,Spanish Empire,sub polity,True
3,Sibylla of Acerra,Kingdom of Sicily,NaN,NaN,none,False
4,Nicola Paglia,Papal States,Holy Roman Empire,Holy Roman Empire,none,False


## Productivity window accuracy

In [6]:
overlap = (comparison[["treccani_end", "cultura_end"]].min(axis=1, skipna=False) - comparison[["treccani_start", "cultura_start"]].max(axis=1, skipna=False) + 1).clip(lower=0)
treccani_span = comparison["treccani_end"] - comparison["treccani_start"] + 1
comparison["overlap_years"] = overlap
comparison["overlap_share"] = (overlap / treccani_span).round(2)
comparison["window_ok"] = comparison["overlap_share"].fillna(0) >= MIN_OVERLAP_SHARE
print(comparison["cultura_start"].isna().sum(), "individual(s) without a Cultura window, counted as wrong")
comparison[["name", "treccani_window", "cultura_window", "overlap_years", "overlap_share", "window_ok"]]

1 individual(s) without a Cultura window, counted as wrong


,name,treccani_window,cultura_window,overlap_years,overlap_share,window_ok
0,Domenico Giunti,1523-1560,1531-1560,30,0.79,True
1,Olimpio Cozzolino,1894-1937,1898-1922,25,0.57,True
2,Restaino Cantelmo Stuart,1675-1721,1690-1707,18,0.38,False
3,Sibylla of Acerra,1190-1204 or 1205,NaN,<NA>,<NA>,False
4,Nicola Paglia,1220-1255,1238-1246,9,0.25,False


## Accuracy of Cultura against Treccani

In [7]:
pd.DataFrame({
    "field": ["polity", "productivity window"],
    "correct": [comparison["polity_ok"].sum(), comparison["window_ok"].sum()],
    "individuals": [len(comparison), len(comparison)],
    "accuracy": [comparison["polity_ok"].mean().round(2), comparison["window_ok"].mean().round(2)],
})

,field,correct,individuals,accuracy
0,polity,2,5,0.4
1,productivity window,2,5,0.4
